In [1]:
import torch
import numpy as np
from sklearn.cluster import MiniBatchKMeans
import pickle
import transformers
import torchaudio
import torchcodec
from torchcodec.decoders import AudioDecoder
from pathlib import Path
from tqdm.notebook import tqdm
import random
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [2]:
#load kmeans_stage2
kmeans_path = '/kaggle/input/datasets/growingsun/kmeans-stage-2-pkl-file/kmeans_stage_2_n=200.pkl'

with open(kmeans_path, 'rb') as file:
    kmeans_stage_2 = pickle.load(file)

In [3]:
#for computing labels for each frame of each chunk, we use the model 9th layer of the model this time
from transformers import HubertModel, Wav2Vec2FeatureExtractor

In [4]:
import requests

try:
    r = requests.get("https://huggingface.co", timeout=10)
    print(r.status_code)
except Exception as e:
    print(e)

200


In [5]:
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained("facebook/hubert-base-ls960")

preprocessor_config.json:   0%|          | 0.00/213 [00:00<?, ?B/s]

In [6]:
output_dir = '/kaggle/working/labels_stage2'
audio_dir = Path('/kaggle/input/datasets/growingsun/cadence-chunks/preprocessed_dir')

In [7]:
from torch.utils.data import Dataset, DataLoader

In [8]:
class FolkAudioDataset(Dataset):
    def __init__(self, audio_dir, label_dir, feature_extractor, kmeans, max_length = 160000):
        self.files = []

        for genre_dir in Path(audio_dir).iterdir():
            if genre_dir.is_dir():
                self.files.extend(list((genre_dir).glob('*')))
        random.shuffle(self.files)

        self.feature_extractor = feature_extractor #wave2vec2 + needs to be stdudied properly
        self.kmeans = kmeans
        self.max_length = max_length
        self.label_dir = label_dir
        
        #mfcc not required
        print(f"Dataset size : {len(self.files)}")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        #retrieve the file with the idx
        file = self.files[idx]

        #decoder the audio and get waveform
        decoder = AudioDecoder(str(file), sample_rate = 16000)
        audio = decoder.get_all_samples()
        waveform = audio.data.mean(dim = 0) #if audio is stereo, converts it to mono
        
        # retrieve the label from the label_dir
        label_path = Path(f'{self.label_dir}/{Path(file).stem}.npy')
        labels = torch.tensor(np.load(label_path), dtype= torch.long)
        
        inputs = self.feature_extractor(
            waveform.numpy(),
            sampling_rate = 16000,
            max_length = self.max_length,
            padding = "max_length",  #all the audioinputs are of same size through padding
            return_tensors = "pt",
            truncation = True
        )

        return inputs.input_values.squeeze(0), labels

In [9]:
labels_dir = '/kaggle/input/datasets/growingsun/labels-stage2'
sample_file = '/kaggle/input/datasets/growingsun/cadence-chunks/preprocessed_dir/lok_dohori/Aaja_Mutu_Bishnu_Majhi_Balaram_Bishwakarma_Shiva_Hamal_New_Nepali_Lok_Dohori_Song_chunk006.wav'

In [10]:
dataset = FolkAudioDataset(audio_dir = audio_dir, label_dir = labels_dir, feature_extractor = feature_extractor, kmeans = kmeans_stage_2)

Dataset size : 11450


In [11]:
dataloader = DataLoader(dataset, batch_size = 4, shuffle= True, num_workers = 4)

In [12]:
def setup_model(checkpoint_path, unfreeze_top_n = 8):
    
    model = HubertModel.from_pretrained(checkpoint_path)
    
    #first freeze all the layers
    for param in model.parameters():
        param.requires_grad = False
    
    total_layers = len(model.encoder.layers)
    # print(total_layers)
    
    for i in range(total_layers - unfreeze_top_n, total_layers):
        for param in model.encoder.layers[i].parameters():
            param.requires_grad = True
    
    #unfreezing the layer norm because the original alpha and beta value, obtained through english sound dataset cannot normalize the nepali songs dataset
    for param in model.encoder.layer_norm.parameters():
        param.requires_grad = True
    
    trainable = sum(p.numel()for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p  in model.parameters())
    print(f"Unfreezed top {unfreeze_top_n} layers")
    print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")
    return model

In [13]:
def create_mask(B, T, device,  mask_length = 10, mask_prob = 0.013):
    """
    B: num_batches = 8
    T: num_timesteps  = 499
    D:  num_dimension of each timestep = 768
    """
    
    mask = torch.zeros(B, T, dtype = torch.bool, device = device)
    num_spans = max(1, int(T * mask_prob)) # 32 positions to be masked

    for b in range(0,B):
    #random mask
        starts = torch.randint(low =0, high = T - mask_length, size=(num_spans,), dtype=torch.int32)   
        for start in starts:
            mask[b, start: start+mask_length] = True
            #mask consecutive span of frames
    
    return mask

In [14]:
def train_stage_2(dataloader,model_checkpoint_path, classifier_checkpoint_path, epochs, batch_size, lr, output_dir, device):
    print(f'device = {device}')

    model = setup_model(model_checkpoint_path) #cpu
    classifier = nn.Linear(768, 200)
    classifier.load_state_dict(
        torch.load(
            classifier_checkpoint_path, map_location = "cpu"
        )
    )

    if model and classifier:
        print(f'Model and classifier checkpoints loaded successfully!!', end='\n')

    model = torch.nn.DataParallel(model).to(device)
    classifier = torch.nn.DataParallel(classifier).to(device)

    #--------------------------------optimizer-----
    optimizer = torch.optim.AdamW(
        list(filter(lambda p: p.requires_grad, model.parameters())) + 
        list(classifier.parameters()),
        lr = lr,
        weight_decay = 0.01)
    
    #-----------------------------------------------------------#
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max = epochs * len(dataloader) 
    ) 
    #T_max  = max num of iterations

    loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

    if not(dataloader or model or classifier or output_dir or device or scheduler or lr or epochs ):
        print(f'All the required parameters are not passed to the train function')
        sys.exit()

    print(f'Dataloader loaded')
    print(f'Oprtimizer loaded')
    print(f'Scheduler loaded')
    print(f'Model loaded')

    # ── Training ───────────────────────────────────────────────────
    epoch_bar = tqdm(range(epochs), desc="Epochs", position=0)
    history   = {'train_loss': [], 'epoch_loss': []}
    
    for epoch in epoch_bar:
        model.train()
        classifier.train()
        total_loss   = 0.0
        total_masked = 0

        step_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs}",
                        position=1, leave=False)

        for step, (input_values, labels) in enumerate(step_bar):
            input_values = input_values.to(device)
            labels = labels.to(device)
    
            optimizer.zero_grad()
            #HuBERT forward pass
            hidden = model(input_values).last_hidden_state  # (B, 499, 768)
            B, T_hubert, D = hidden.shape
    
            # Create mask
            mask = create_mask(B, T_hubert, device)   # (B, 499)

            # No interpolation needed ← labels already 499 frames
            
            # Extract masked positions
            masked_hidden = hidden[mask]              # (N, 768)
            masked_labels = labels[mask]              # (N,)

            # Filter invalid labels
            valid         = (masked_labels >= 0) & (masked_labels < 200)  # ← 200
            masked_hidden = masked_hidden[valid]
            masked_labels = masked_labels[valid]

            if masked_labels.numel() == 0:
                continue
                
            # Predict and compute loss
            logits = classifier(masked_hidden)        # (N, 200)
            loss   = loss_fn(logits, masked_labels)
            
            # Backprop
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

            total_loss   += loss.item()
            total_masked += mask.sum().item()
            history['train_loss'].append(loss.item())

            step_bar.set_postfix({
                "loss"   : f"{loss.item():.4f}",
                "masked" : mask.sum().item(),
                "lr"     : f"{scheduler.get_last_lr()[0]:.2e}"
            })
            
        avg_loss = total_loss / len(dataloader)
        history['epoch_loss'].append(avg_loss)
        print(f'Average_loss = {avg_loss}',end='\n')

        epoch_bar.set_postfix({
            "avg_loss"     : f"{avg_loss:.4f}",
            "total_masked" : f"{total_masked:,}"
        })

        # Save checkpoint every 5 epochs
        if (epoch + 1 + 15) % 5 == 0:
            ckpt = f"{output_dir}/stage2-checkpoint-epoch{epoch+1+15}"
            model.module.save_pretrained(ckpt)
            feature_extractor.save_pretrained(ckpt)
            torch.save(classifier.module.state_dict(), f"{ckpt}/classifier.pt")
            tqdm.write(f"Checkpoint saved → {ckpt}")
        
    model.module.save_pretrained(f"{output_dir}/hubert-folk-stage2")
    feature_extractor.save_pretrained(f"{output_dir}/hubert-folk-stage2")
    torch.save(classifier.module.state_dict(),
               f"{output_dir}/hubert-folk-stage2/classifier.pt")
    tqdm.write("Stage 2 training complete.")
    return history

In [15]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [16]:
import gc
torch.cuda.empty_cache()
gc.collect()

0

In [17]:
model_checkpoint_path = '/kaggle/input/datasets/growingsun/checkpoint-epoch15-hubert-stage2'
classifier_path = '/kaggle/input/datasets/growingsun/checkpoint-epoch15-hubert-stage2/classifier.pt'
output_dir = '/kaggle/working/'
batch_size = 4
lr = 5e-6
epochs = 5

In [18]:
history = train_stage_2(dataloader,model_checkpoint_path, classifier_path, epochs, batch_size, lr, output_dir, device)

device = cuda


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Unfreezed top 8 layers
Trainable: 56,704,512 / 94,371,712 (60.1%)
Model and classifier checkpoints loaded successfully!!
Dataloader loaded
Oprtimizer loaded
Scheduler loaded
Model loaded


Epochs:   0%|          | 0/5 [00:00<?, ?it/s]

Epoch 1/5:   0%|          | 0/2863 [00:00<?, ?it/s]

...

Average_loss = 1.007860107081277


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Checkpoint saved → /kaggle/working//stage2-checkpoint-epoch20


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Stage 2 training complete.
